# Wikipedia Tool

In [21]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

import wikipedia

# Set a custom user agent to avoid being blocked by Wikipedia
wikipedia.set_user_agent("langchain-course-example/1.0 (learning project)")

api_wapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
tool = WikipediaQueryRun(api_wrapper=api_wapper)

tool.run("langchain")

"Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis."

# Search Tool

In [22]:
from dotenv import load_dotenv, find_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults

load_dotenv(find_dotenv(usecwd=True))

True

In [23]:
search = TavilySearchResults(max_results=2)

In [24]:
search.invoke("¿Qué es LangChain?")

[{'title': '¿Qué es LangChain? Ejemplos y definición | Google Cloud',
  'url': 'https://cloud.google.com/use-cases/langchain?hl=es-419',
  'content': '# ¿Qué es LangChain?\n\nLangChain es un framework de organización de código abierto que simplifica la creación de aplicaciones con modelos de lenguaje grandes (LLM). Proporciona herramientas y componentes para conectar LLM con varias fuentes de datos, lo que permite la creación de flujos de trabajo complejos de varios pasos.\n\nLangChain, disponible como biblioteca en Python y JavaScript, ayuda a los desarrolladores a mejorar las capacidades de los LLM más allá de la generación de texto vinculándolos a datos externos y computación. Esto ayuda a facilitar el desarrollo de aplicaciones de IA avanzadas, como chatbots inteligentes, sistemas sofisticados de respuesta a preguntas y herramientas automatizadas de análisis de datos.\n\nComenzar gratis',
  'score': 0.9483129},
 {'title': '¿Qué es LangChain?: Explicación sobre LangChain: AWS',
  'u

In [25]:
tools = [search, tool]

In [26]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="stealth/crouton-alpha",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0 # no invente nada
)

In [27]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# prompt escrito a mano: mismo esquema del prompt "hwchase17/openai-functions-agent"
# sin bajarlo del hub (LangSmith bloquea prompts públicos por defecto)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant, usarás las tools que tienes a tu disposición para responder a las preguntas de los usuarios."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant, usarás las tools que tienes a tu disposición para responder a las preguntas de los usuarios.'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

# Agents 

In [28]:
from langchain_classic.agents import create_tool_calling_agent

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

In [29]:
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [30]:
agent_executor.invoke(input={"input": "Hola"})



> Entering new AgentExecutor chain...
¡Hola! Soy Crouton Alpha. ¿En qué puedo ayudarte hoy?

> Finished chain.


{'input': 'Hola',
 'output': '¡Hola! Soy Crouton Alpha. ¿En qué puedo ayudarte hoy?'}

In [31]:
agent_executor.invoke(input={"input":"¿Cual es el mejor superheroe de Marvel y por que?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'mejor superheroe de Marvel ranking popularidad'}`


[{'title': 'Marvel Comics | Popular Characters | Comic Cruncher', 'url': 'https://comiccruncher.com/marvel', 'content': 'Comic Cruncher\n\n# Marvel Comics\n\n# Marvel Characters Ranked By Appearances\n\nThis page shows Marvel characters ranked by number of main reality appearances (no alternate realities).\n\n#1.Spider-Man\n\n4841issues\n\n#2.Captain America\n\n3940issues\n\n#3.Iron Man\n\n3631issues\n\n#4.Wolverine\n\n3590issues\n\n#5.Thor\n\n2610issues\n\n#6.Thing\n\n2545issues\n\n#7.Hulk\n\n2340issues\n\n#8.Mr. Fantastic\n\n2283issues\n\n#9.Cyclops\n\n2267issues\n\n#10.Beast\n\n2210issues\n\n#11.Human Torch\n\n2191issues\n\n#12.Storm\n\n1882issues\n\n#13.Invisible Woman\n\n1864issues\n\n#14.Hawkeye\n\n1750issues\n\n#15.Doctor Strange\n\n1727issues\n\n#16.Namor\n\n1704issues\n\n#17.Daredevil\n\n1622issues\n\n#18.Iceman\n\n1510issues\n\n#

{'input': '¿Cual es el mejor superheroe de Marvel y por que?',
 'output': 'No hay un "mejor" objetivo, porque depende de qué busques: poder, historia, carisma o identificación. Pero si hay que elegir uno por consenso general —en cómics, cine y popularidad— **Spider-Man (Peter Parker) es el que casi siempre gana.**\n\nY no es solo por los datos: según el ranking de apariciones en Marvel, Spider-Man es el #1 con 4.841 números en la continuidad principal, por encima de Captain America (3.940) e Iron Man (3.631).\n\n### ¿Por qué Spider-Man?\n\n**1. Es el más humano**\nA diferencia de Thor (un dios) o Iron Man (un genio multimillonario), Peter es un chico normal de Queens con problemas normales: dinero, estudios, trabajo, familia. Sus superpoderes no resuelven su vida, a veces la complican más.\n\n**2. Su lema lo define: "Un gran poder conlleva una gran responsabilidad"**\nEs el héroe que más sacrifica por hacer lo correcto, incluso cuando nadie lo ve o cuando le cuesta todo. Esa es la base

In [32]:
agent_executor.invoke(input={"input":"¿Que temperatura hace actualmente en tegucigalpa honduras?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'temperatura actual Tegucigalpa Honduras ahora'}`


[{'title': 'Tiempo en Tegucigalpa - Meteored Honduras', 'url': 'https://www.meteored.hn/tiempo-en_Tegucigalpa-America+Central-Honduras-Francisco+Morazan--1-21041.html', 'content': 'Hoy en Tegucigalpa, con temperaturas alrededor de 23°C. temperaturas en torno a los 26°C. temperaturas cercanas a los 21°C. 28° / 18°', 'score': 0.9051298}, {'title': 'Pronóstico meteorológico de Tegucigalpa, Francisco-Morazán | MSN El Tiempo', 'url': 'https://www.msn.com/es-us/el-tiempo/pronostico/in-Tegucigalpa,Francisco-Moraz%C3%A1n', 'content': '# Tegucigalpa, Francisco Morazán, Honduras\n\nEl tiempo actual\n\n7:36 PM\n\n26°C\n\nParc. nublado\n\nSensación térmica\n\n\u200e25°\n\nEl cielo estará parcialmente nublado. La temperatura mínima será de 21°.\n\nCalidad del aire\n\nCalidad del aire moderada (51-100)  \n Contaminante principal PM2.5 2.5 μg/m³\n\n64\n\n

{'input': '¿Que temperatura hace actualmente en tegucigalpa honduras?',
 'output': 'Actualmente en **Tegucigalpa, Honduras** hace **26°C**.\n\nSegún el reporte más reciente (7:36 PM):\n\n* **Condición:** Parcialmente nublado\n* **Sensación térmica:** 25°C\n* **Humedad:** 74%\n* **Viento:** 10 km/h, brisa ligera\n* **Temperatura mínima prevista para hoy:** 21°C'}